# 01 · Preprocess: windowing
Discovers ambient, attack and synthetic captures for the selected dataset and cuts them into 2.0 s windows (1.0 s stride). Runs `graph-transformer/preprocess.py`.

In [ ]:
import os
import sys

# Pick the dataset BEFORE importing pipeline modules: "road" (default) or a converted
# can-train-and-test set such as "ctt_set_01" (see graph-transformer/convert_can_train_and_test.py).
os.environ["CAN_DATASET"] = os.environ.get("CAN_DATASET", "road")

# Repository root = first parent that holds graph-transformer/. All logic lives in the canonical
# modules; this notebook only calls them, so it cannot drift from the scripts.
REPO = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(REPO, "graph-transformer")) and os.path.dirname(REPO) != REPO:
    REPO = os.path.dirname(REPO)
for d in ("explainability", "attack-detection", "graph-transformer"):
    p = os.path.join(REPO, d)
    if p not in sys.path:
        sys.path.insert(0, p)
print("repo:", REPO, "| dataset:", os.environ["CAN_DATASET"])

For can-train-and-test, convert the raw set once first (writes `the second/converted/<dataset>/`).

In [ ]:
DATASET = os.environ["CAN_DATASET"]
if DATASET.startswith("ctt_"):
    import convert_can_train_and_test
    if not os.path.isdir(os.path.join(REPO, "the second", "converted", DATASET)):
        convert_can_train_and_test.main(DATASET.removeprefix("ctt_"))

In [ ]:
import preprocess

windows = preprocess.build_dataset(keep_messages=False)
preprocess.save_dataset(windows, preprocess.OUTPUT_DIR)

In [ ]:
import collections

kinds = collections.Counter(
    ("synthetic" if w.capture_name.startswith("synthetic__") else "ambient" if w.capture_name.startswith("ambient_") else "attack",
     w.label) for w in windows)
for (kind, label), n in sorted(kinds.items()):
    print(f"{kind:10s} label={label}: {n:6d} windows")